<a href="https://colab.research.google.com/github/jringler30/finance-python-models/blob/main/DDM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#### Libraries ####
import yfinance as yf
from datetime import date


def yearfrac(end_date: date, start_date: date | None = None) -> float:
    """Return fractional years between two dates using a 365-day basis."""
    start = start_date or date.today()
    return (end_date - start).days / 365


def get_stock_price(symbol: str) -> float:
    stock = yf.Ticker(symbol)
    price = stock.fast_info.get("last_price")
    if price is None:
        raise ValueError(f"Unable to fetch current price for {symbol}.")
    return round(float(price), 2)


#### Inputs ####
base_year = date.today().year
discount_rate = float(input("What is the discount rate? (decimal, e.g. 0.10): "))
forecasted_periods = int(input("How many periods are we forecasting?: "))
symbol = input("What is the stock ticker?: ").upper().strip()
terminal_growth_rate = float(input("What is the terminal growth rate? (decimal): "))

if discount_rate <= terminal_growth_rate:
    raise ValueError("Discount rate must be greater than terminal growth rate.")
if forecasted_periods <= 0:
    raise ValueError("Forecasted periods must be >= 1.")

price = get_stock_price(symbol)

#### Forecasted Dividends ####
forecasted_dividends = []
for i in range(forecasted_periods):
    dividend = float(input(f"What is the dividend for period {i + 1}?: "))
    forecasted_dividends.append(dividend)

#### Terminal Value ####
terminal_value = (forecasted_dividends[-1] * (1 + terminal_growth_rate)) / (discount_rate - terminal_growth_rate)
terminal_date = date(base_year + forecasted_periods - 1, 12, 31)
terminal_value_present_value = terminal_value / ((1 + discount_rate) ** yearfrac(terminal_date))
terminal_value_present_value = round(terminal_value_present_value, 2)

#### Present Value of Dividends ####
present_value_dividends = 0.0
for i, dividend in enumerate(forecasted_dividends):
    dividend_date = date(base_year + i, 12, 31)
    present_value_dividends += dividend / ((1 + discount_rate) ** yearfrac(dividend_date))

present_value_dividends = round(present_value_dividends, 2)

#### Net Present Value ####
net_present_value = round(present_value_dividends + terminal_value_present_value, 2)

print(f"The price of {symbol} is ${price}")
print(f"The Net Present Value of the stock is ${net_present_value}")

if net_present_value > price * 1.05:
    print(f"{symbol} is worth investing in")
elif net_present_value < price * 0.95:
    print(f"{symbol} is not worth investing in")
else:
    print(f"Hold {symbol}")


What is the discount rate?: 0.0525
How many periods are we forecasting?: 5
What is the stock ticker?: wmt
What is the terminal growth rate?: 0.035
What is the dividend for period 1?: 0.91
What is the dividend for period 2?: 1.05
What is the dividend for period 3?: 1.3
What is the dividend for period 4?: 1.55
What is the dividend for period 5?: 1.82
85.83
0.37574511843318065
1.3520631108548513
2.5005425952101703
3.801396726511549
5.252659936361222
The price of WMT is $97.59
The Net Present Value of the stock is $91.08
WMT is not worth investing in
